Курсовая работа Витрик Григорий ИД24-1 2.2

Анализ характеристик музыкальных треков для поиска паттернов популярности и жанровой принадлежности

##Основной датасет MTG-Jamendo

In [2]:
!git clone https://github.com/MTG/mtg-jamendo-dataset.git

Cloning into 'mtg-jamendo-dataset'...
remote: Enumerating objects: 1290, done.
remote: Counting objects: 100% (260/260), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 1290 (delta 246), reused 228 (delta 226), pack-reused 1030 (from 1)
Receiving objects: 100% (1290/1290), 42.37 MiB | 17.03 MiB/s, done.
Resolving deltas: 100% (792/792), done.
Updating files: 100% (186/186), done.


In [3]:
%cd mtg-jamendo-dataset

/content/mtg-jamendo-dataset


In [4]:
!ls

audio_licenses.txt  _config.yml  derived  README.md  scripts  stats
CITATION.bib	    data	 LICENSE  results    split


In [5]:
!ls data

autotagging_genre.tsv	    download			     raw.tsv
autotagging_instrument.tsv  raw_30s_cleantags_50artists.tsv  splits
autotagging_moodtheme.tsv   raw_30s_cleantags.tsv	     tag_map.json
autotagging_top50tags.tsv   raw_30s.tsv			     tags
autotagging.tsv		    raw.meta.tsv


In [6]:
import os
import pandas as pd
import requests
import xml.etree.ElementTree as ET

In [7]:
rows = []

with open("data/autotagging_instrument.tsv", "r", encoding="utf-8") as f:
  for line in f:
    if line.startswith("#"):
      continue

    parts = line.strip().split("\t")

    rows.append(parts)

rows[:3]

[['TRACK_ID', 'ARTIST_ID', 'ALBUM_ID', 'PATH', 'DURATION', 'TAGS'],
 ['track_0000382',
  'artist_000020',
  'album_000046',
  '82/382.mp3',
  '211.1',
  'instrument---voice'],
 ['track_0000383',
  'artist_000020',
  'album_000046',
  '83/383.mp3',
  '113.1',
  'instrument---voice']]

In [8]:
data = []

for parts in rows[1:]:
    data.append({
        "track_id": parts[0],
        "artist_id": parts[1],
        "album_id": parts[2],
        "path": parts[3],
        "duration": float(parts[4]),
        "tags": parts[5:]
    })

df_instruments = pd.DataFrame(data)

df_instruments.head()

,track_id,artist_id,album_id,path,duration,tags
0,track_0000382,artist_000020,album_000046,82/382.mp3,211.1,[instrument---voice]
1,track_0000383,artist_000020,album_000046,83/383.mp3,113.1,[instrument---voice]
2,track_0000384,artist_000020,album_000046,84/384.mp3,115.7,[instrument---voice]
3,track_0000386,artist_000020,album_000046,86/386.mp3,103.4,[instrument---voice]
4,track_0000387,artist_000020,album_000046,87/387.mp3,257.1,[instrument---voice]


In [9]:
df_instruments["instruments"] = df_instruments["tags"].apply(
  lambda x: [tag.replace("instrument---", "") for tag in x if tag.startswith("instrument---")])

df_instruments = df_instruments.drop(columns=["tags"])

In [10]:
df_instruments.head(50)

,track_id,artist_id,album_id,path,duration,instruments
0,track_0000382,artist_000020,album_000046,82/382.mp3,211.1,[voice]
1,track_0000383,artist_000020,album_000046,83/383.mp3,113.1,[voice]
2,track_0000384,artist_000020,album_000046,84/384.mp3,115.7,[voice]
3,track_0000386,artist_000020,album_000046,86/386.mp3,103.4,[voice]
4,track_0000387,artist_000020,album_000046,87/387.mp3,257.1,[voice]
5,track_0000388,artist_000020,album_000046,88/388.mp3,124.6,[voice]
6,track_0000389,artist_000020,album_000046,89/389.mp3,150.8,[voice]
7,track_0000390,artist_000020,album_000046,90/390.mp3,133.7,[voice]
8,track_0000391,artist_000020,album_000046,91/391.mp3,82.1,[voice]
9,track_0000392,artist_000020,album_000046,92/392.mp3,127.3,[voice]


In [11]:
df_instruments.shape

(25135, 6)

In [12]:
df_instruments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25135 entries, 0 to 25134
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   track_id     25135 non-null  object 
 1   artist_id    25135 non-null  object 
 2   album_id     25135 non-null  object 
 3   path         25135 non-null  object 
 4   duration     25135 non-null  float64
 5   instruments  25135 non-null  object 
dtypes: float64(1), object(5)
memory usage: 1.2+ MB


In [13]:
df_instruments.isna().sum()

,0
track_id,0
artist_id,0
album_id,0
path,0
duration,0
instruments,0


In [15]:
!ls data | grep genre

autotagging_genre.tsv


In [16]:
from collections import Counter

instrument_counts = Counter(
    instrument
    for instruments in df_instruments["instruments"]
    for instrument in instruments
)

pd.DataFrame(
    instrument_counts.items(),
    columns=["instrument", "count"]
).sort_values("count", ascending=False).head(20)

,instrument,count
2,piano,7750
1,synthesizer,7496
18,drums,6120
15,bass,5726
5,guitar,4804
19,electricguitar,4241
16,computer,2080
7,keyboard,2044
21,acousticguitar,1947
13,violin,1837


In [17]:
rows = []

with open("data/autotagging_genre.tsv", "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.strip().split("\t")
        rows.append(parts)

data = []

for parts in rows[1:]:
    data.append({
        "track_id": parts[0],
        "artist_id": parts[1],
        "album_id": parts[2],
        "path": parts[3],
        "duration": float(parts[4]),
        "genres": [
            tag.replace("genre---", "")
            for tag in parts[5:]
            if tag.startswith("genre---")
        ]
    })

df_genres = pd.DataFrame(data)

In [18]:
df_genres.head()

,track_id,artist_id,album_id,path,duration,genres
0,track_0000214,artist_000014,album_000031,14/214.mp3,124.6,[punkrock]
1,track_0000215,artist_000014,album_000031,15/215.mp3,151.4,[metal]
2,track_0000216,artist_000014,album_000031,16/216.mp3,234.9,[metal]
3,track_0000217,artist_000014,album_000031,17/217.mp3,127.9,[punkrock]
4,track_0000218,artist_000014,album_000031,18/218.mp3,180.7,[punkrock]


In [19]:
df_genres.shape

(55215, 6)

In [20]:
df_genres.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55215 entries, 0 to 55214
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   track_id   55215 non-null  object 
 1   artist_id  55215 non-null  object 
 2   album_id   55215 non-null  object 
 3   path       55215 non-null  object 
 4   duration   55215 non-null  float64
 5   genres     55215 non-null  object 
dtypes: float64(1), object(5)
memory usage: 2.5+ MB


In [21]:
df_genres.isna().sum()

,0
track_id,0
artist_id,0
album_id,0
path,0
duration,0
genres,0


In [22]:
all_genres = sorted({
    genre
    for genres in df_genres["genres"]
    for genre in genres
})

print("Количество жанров:", len(all_genres))
print(all_genres)

Количество жанров: 95
['60s', '70s', '80s', '90s', 'acidjazz', 'african', 'alternative', 'alternativerock', 'ambient', 'atmospheric', 'blues', 'bluesrock', 'bossanova', 'breakbeat', 'celtic', 'chanson', 'chillout', 'choir', 'classical', 'classicrock', 'club', 'contemporary', 'country', 'dance', 'darkambient', 'darkwave', 'deephouse', 'disco', 'downtempo', 'drumnbass', 'dub', 'dubstep', 'easylistening', 'edm', 'electronic', 'electronica', 'electropop', 'ethnicrock', 'ethno', 'eurodance', 'experimental', 'folk', 'funk', 'fusion', 'gothic', 'groove', 'grunge', 'hard', 'hardrock', 'heavymetal', 'hiphop', 'house', 'idm', 'improvisation', 'indie', 'industrial', 'instrumentalpop', 'instrumentalrock', 'jazz', 'jazzfunk', 'jazzfusion', 'latin', 'lounge', 'medieval', 'metal', 'minimal', 'newage', 'newwave', 'orchestral', 'oriental', 'pop', 'popfolk', 'poprock', 'postrock', 'progressive', 'psychedelic', 'punkrock', 'rap', 'reggae', 'rnb', 'rock', 'rocknroll', 'singersongwriter', 'ska', 'soul', 's

In [23]:
genre_counts = Counter(
    genre
    for genres in df_genres["genres"]
    for genre in genres
)

pd.DataFrame(
    genre_counts.items(),
    columns=["genre", "count"]
).sort_values("count", ascending=False).head(20)

,genre,count
6,electronic,16480
30,soundtrack,8094
3,pop,7805
12,ambient,7570
2,rock,6865
4,classical,5602
7,easylistening,4833
24,experimental,3941
23,alternative,3761
13,chillout,3678


In [24]:
rows = []

with open("data/autotagging_moodtheme.tsv", "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.strip().split("\t")
        rows.append(parts)

data = []

for parts in rows[1:]:
    data.append({
        "track_id": parts[0],
        "artist_id": parts[1],
        "album_id": parts[2],
        "path": parts[3],
        "duration": float(parts[4]),
        "moodtheme": [
            tag.replace("mood/theme---", "")
            for tag in parts[5:]
            if tag.startswith("mood/theme---")
        ]
    })

df_mood = pd.DataFrame(data)

In [25]:
df_mood.head()

,track_id,artist_id,album_id,path,duration,moodtheme
0,track_0000948,artist_000087,album_000149,48/948.mp3,212.7,[background]
1,track_0000950,artist_000087,album_000149,50/950.mp3,248.0,[background]
2,track_0000951,artist_000087,album_000149,51/951.mp3,199.7,[background]
3,track_0002165,artist_000326,album_000347,65/2165.mp3,229.0,[film]
4,track_0002263,artist_000320,album_000366,63/2263.mp3,494.7,[melancholic]


In [26]:
df_mood.shape

(18486, 6)

In [27]:
df_mood.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18486 entries, 0 to 18485
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   track_id   18486 non-null  object 
 1   artist_id  18486 non-null  object 
 2   album_id   18486 non-null  object 
 3   path       18486 non-null  object 
 4   duration   18486 non-null  float64
 5   moodtheme  18486 non-null  object 
dtypes: float64(1), object(5)
memory usage: 866.7+ KB


In [28]:
df_mood.isna().sum()

,0
track_id,0
artist_id,0
album_id,0
path,0
duration,0
moodtheme,0


In [29]:
df_tracks = df_genres.merge(
    df_instruments[["track_id", "instruments"]],
    on="track_id",
    how="inner"
)

df_tracks = df_tracks.merge(
    df_mood[["track_id", "moodtheme"]],
    on="track_id",
    how="left"
)

df_tracks.head()

,track_id,artist_id,album_id,path,duration,genres,instruments,moodtheme
0,track_0000382,artist_000020,album_000046,82/382.mp3,211.1,[classical],[voice],NaN
1,track_0000383,artist_000020,album_000046,83/383.mp3,113.1,[classical],[voice],NaN
2,track_0000384,artist_000020,album_000046,84/384.mp3,115.7,[classical],[voice],NaN
3,track_0000386,artist_000020,album_000046,86/386.mp3,103.4,[classical],[voice],NaN
4,track_0000387,artist_000020,album_000046,87/387.mp3,257.1,[classical],[voice],NaN


In [30]:
df_tracks.shape

(24894, 8)

In [31]:
df_tracks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24894 entries, 0 to 24893
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   track_id     24894 non-null  object 
 1   artist_id    24894 non-null  object 
 2   album_id     24894 non-null  object 
 3   path         24894 non-null  object 
 4   duration     24894 non-null  float64
 5   genres       24894 non-null  object 
 6   instruments  24894 non-null  object 
 7   moodtheme    10516 non-null  object 
dtypes: float64(1), object(7)
memory usage: 1.5+ MB


In [32]:
df_tracks.isna().sum()

,0
track_id,0
artist_id,0
album_id,0
path,0
duration,0
genres,0
instruments,0
moodtheme,14378


Также MTG-Jamendo содержит ссылки на соответствующие аудиофайлы треков.
Аудиозаписи относятся к неструктурированным данным и могут использоваться
для последующего извлечения дополнительных акустических признаков.

In [49]:
df_tracks[["track_id", "path"]].head(10)

,track_id,path
0,track_0000382,82/382.mp3
1,track_0000383,83/383.mp3
2,track_0000384,84/384.mp3
3,track_0000386,86/386.mp3
4,track_0000387,87/387.mp3
5,track_0000388,88/388.mp3
6,track_0000389,89/389.mp3
7,track_0000390,90/390.mp3
8,track_0000391,91/391.mp3
9,track_0000392,92/392.mp3


##MusicBrainz (XML)

In [33]:
url = "https://musicbrainz.org/ws/2/recording/"
params = {
    "query": 'recording:"Yellow" AND artist:"Coldplay"',
    "fmt": "xml"
}

headers = {
    "User-Agent": "MusicCoursework/1.0"
}

response = requests.get(url, params=params, headers=headers)

print(response.status_code)

200


In [34]:
print(response.text[:2000])

<?xml version="1.0" encoding="UTF-8" standalone="yes"?><metadata created="2026-10-09T14:34:55.103Z" xmlns="http://musicbrainz.org/ns/mmd-2.0#" xmlns:ns2="http://musicbrainz.org/ns/ext#-2.0"><recording-list count="146" offset="0"><recording id="4352d312-6e75-42a4-bf0d-574abb922719" ns2:score="100"><title>Yellow</title><length>329000</length><disambiguation>live, 2003-07-22: Horden Pavillion, Sydney, Australia</disambiguation><artist-credit id="d97213b4-e672-3cde-9e53-6870e168f6ae"><name-credit><name>Coldplay</name><artist id="cc197bad-dc9c-440d-a5b5-d52ba2e14234"><name>Coldplay</name><sort-name>Coldplay</sort-name><alias-list><alias sort-name="Coolplay" type="Search hint" type-id="1937e404-b981-3cb7-8151-4c86ebfc8d8e">Coolplay</alias><alias sort-name="콜드플레이" type="Search hint" type-id="1937e404-b981-3cb7-8151-4c86ebfc8d8e">콜드플레이</alias><alias locale="ja" sort-name="コールドプレイ" type="Artist name" type-id="894afba6-2816-3c24-8072-eadb66bd04bc">コールドプレイ</alias><alias sort-name="Cold Play" type

In [35]:
root = ET.fromstring(response.text)
root.tag

'{http://musicbrainz.org/ns/mmd-2.0#}metadata'

In [36]:
ns = {
    "m": "http://musicbrainz.org/ns/mmd-2.0#"
}

recordings = root.findall(".//m:recording", ns)

len(recordings)

25

In [37]:
recordings[0].find("m:title", ns).text

'Yellow'

In [38]:
musicbrainz_rows = []

for recording in recordings:
    title = recording.find("m:title", ns)

    musicbrainz_rows.append({
        "recording_id": recording.attrib.get("id"),
        "title": title.text if title is not None else None
    })

df_musicbrainz = pd.DataFrame(musicbrainz_rows)

In [39]:
df_musicbrainz.head(25)

,recording_id,title
0,4352d312-6e75-42a4-bf0d-574abb922719,Yellow
1,2e30d825-55ad-4d8c-a527-50e4a1909568,Yellow
2,a33d09b1-7ee3-4077-9c56-597dee535434,Yellow
3,98f07927-221f-452f-bf5d-65340849574d,Yellow
4,da97b6cb-68ff-4f2a-b726-29c8ddb2668e,Yellow
5,e663d2b8-1309-4f80-8495-1fb17e23f9c6,Yellow
6,1e75db0a-bb43-458c-8cc3-dbd997422bc1,Yellow
7,507b12aa-9ee9-4aa8-a741-b4edaa1d826e,Yellow
8,935b8767-ec6f-4a10-9987-1ebdaf068fe9,Yellow
9,447f9d36-7754-43ae-9c7f-3e0fd9e29a8b,Yellow


In [40]:
df_musicbrainz.shape

(25, 2)

In [41]:
df_musicbrainz.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   recording_id  25 non-null     object
 1   title         25 non-null     object
dtypes: object(2)
memory usage: 532.0+ bytes


##Last.fm (API)

In [42]:
LASTFM_API_KEY = '''api_key'''

url = "https://ws.audioscrobbler.com/2.0/"

params = {
    "method": "track.getInfo",
    "api_key": LASTFM_API_KEY,
    "artist": "Coldplay",
    "track": "Yellow",
    "format": "json"
}

response = requests.get(url, params=params)

print(response.status_code)

200


In [43]:
data = response.json()
data

{'track': {'name': 'Yellow',
  'mbid': '0063bb06-4e74-464d-97ae-b82f90bcb222',
  'url': 'https://www.last.fm/music/Coldplay/_/Yellow',
  'duration': '266000',
  'streamable': {'#text': '0', 'fulltrack': '0'},
  'listeners': '3665825',
  'playcount': '40432963',
  'artist': {'name': 'Coldplay',
   'mbid': 'cc197bad-dc9c-440d-a5b5-d52ba2e14234',
   'url': 'https://www.last.fm/music/Coldplay'},
  'album': {'artist': 'Coldplay',
   'title': 'Parachutes',
   'url': 'https://www.last.fm/music/Coldplay/Parachutes',
   'image': [{'#text': 'https://lastfm-img.freetls.fastly.net/i/u/34s/5fbcfb7258117e88cb73b45e25c88b01.png',
     'size': 'small'},
    {'#text': 'https://lastfm-img.freetls.fastly.net/i/u/64s/5fbcfb7258117e88cb73b45e25c88b01.png',
     'size': 'medium'},
    {'#text': 'https://lastfm-img.freetls.fastly.net/i/u/174s/5fbcfb7258117e88cb73b45e25c88b01.png',
     'size': 'large'},
    {'#text': 'https://lastfm-img.freetls.fastly.net/i/u/300x300/5fbcfb7258117e88cb73b45e25c88b01.png',
  

In [44]:
track = data["track"]

print("Название:", track["name"])
print("Исполнитель:", track["artist"]["name"])
print("Listeners:", track["listeners"])
print("Playcount:", track["playcount"])

Название: Yellow
Исполнитель: Coldplay
Listeners: 3665825
Playcount: 40432963


In [45]:
tags = [tag["name"] for tag in track["toptags"]["tag"]]
tags

['rock', 'alternative', 'Coldplay', 'alternative rock', 'britpop']

In [46]:
df_lastfm = pd.DataFrame([{
    "track": track["name"],
    "artist": track["artist"]["name"],
    "listeners": int(track["listeners"]),
    "playcount": int(track["playcount"]),
    "tags": tags
}])

df_lastfm

,track,artist,listeners,playcount,tags
0,Yellow,Coldplay,3665825,40432963,"[rock, alternative, Coldplay, alternative rock..."




1. **MTG-Jamendo** — табличные данные о музыкальных треках,
   включающие длительность, жанры, инструменты и mood/theme.
   Таблицы связаны по идентификатору `track_id`.

2. **MusicBrainz API** — источник иерархических данных в формате XML.
   Используется для получения метаданных о музыкальных записях.

3. **Last.fm API** — источник данных в формате JSON.
   Позволяет получать количество слушателей, количество прослушиваний
   и пользовательские теги музыкальных треков.

Числовые характеристики: длительность, listeners, playcount.

Категориальные характеристики: жанр, инструменты, mood/theme,
исполнитель и пользовательские теги.